# 🚶 Explorar — paseos bonitos por un documento
Sin entrenar nada: cargar un PDF y MIRARLO con los mismos ojos que usa el sistema.
Casi todo es determinista (sin LLM); solo el mapa semantico necesita la key de OpenAI.


In [ ]:
import sys, os, json, pathlib, re
import numpy as np
import matplotlib.pyplot as plt

# 1) API keys ANTES de importar rag_lab (CONFIG es una instantanea al importar)
# os.environ['OPENAI_API_KEY'] = '...'
# os.environ['ANTHROPIC_API_KEY'] = '...'
# os.environ['VOYAGE_API_KEY'] = '...'   # opcional
# os.environ['COHERE_API_KEY'] = '...'   # opcional

# 2) Ruta del proyecto: HARDCODEADA (en local, cambia esta linea a tu ruta)
RAIZ = pathlib.Path('/content/drive/MyDrive/rag_lab/rag_lab_proyecto')
sys.path.insert(0, str(RAIZ))

from rag_lab.config import CONFIG
from rag_lab.memory import Memoria
mem = Memoria(CONFIG.fichero_memoria)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})
print('Memoria:', CONFIG.fichero_memoria)
print('Resumen:', mem.resumen())


## 1. El perfil: la ficha del documento sin ningun LLM


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from rag_lab.perfil import perfil_de_documento, descriptor, NUMERICAS
PDF = RAIZ.parent / 'pdfs' / 'MI_DOCUMENTO.pdf'   # <-- ajusta
docs = PyPDFLoader(str(PDF)).load()
perfil = perfil_de_documento(docs)
print(descriptor(perfil)); perfil


## 2. El radar del perfil
Las features en un vistazo (la silueta se anade tras el paso 4 si lo corres).


In [ ]:
feats = [k for k in NUMERICAS if perfil.get(k) is not None]
vals = [max(0.0, float(perfil[k])) for k in feats]
ang = np.linspace(0, 2*np.pi, len(feats), endpoint=False).tolist()
fig, ax = plt.subplots(figsize=(5.5, 5.5), subplot_kw={'projection': 'polar'})
ax.plot(ang + ang[:1], vals + vals[:1], lw=2); ax.fill(ang + ang[:1], vals + vals[:1], alpha=0.25)
ax.set_xticks(ang); ax.set_xticklabels(feats, fontsize=8)
ax.set_title(f'Perfil: {descriptor(perfil)}'); plt.show()


## 3. ¿De que habla? — los temas LDA
Las palabras que definen cada tema. La dispersion baja = documento mono-tema (y empates probables).


In [ ]:
from rag_lab.analisis import analizar_temas
lda = analizar_temas(docs)
if lda.get('disponible'):
    print('dispersion tematica:', lda['dispersion_tematica'])
    fig, ejes = plt.subplots(1, lda['n_temas'], figsize=(3.4 * lda['n_temas'], 3.4))
    for ax, tema in zip(np.atleast_1d(ejes), lda['temas']):
        palabras = tema['palabras'][:8][::-1]
        ax.barh(palabras, range(1, len(palabras) + 1), alpha=0.75)
        ax.set_title(f"tema {tema['tema']}", fontsize=10); ax.set_xticks([])
    plt.suptitle('¿De que habla el documento?'); plt.tight_layout(); plt.show()
else:
    print('LDA no disponible:', lda.get('motivo'))


## 4. El mapa semantico y el hilo narrativo *(usa OpenAI)*
Cada punto es un fragmento; el color, su orden en el documento. Un degradado limpio = el texto
AVANZA por el espacio semantico (estructura narrativa); una nube revuelta = temas entremezclados.


In [ ]:
from rag_lab.supervisor.plan import pipeline_base
from rag_lab.bancos.embeddings_banco import _construir_splitter
from rag_lab.fabricas import construir_embedding
from rag_lab.analisis.umap_proyeccion import proyectar_embeddings
base = pipeline_base()
chunks = _construir_splitter(base['splitter']).split_documents(docs)
proy = proyectar_embeddings(chunks, construir_embedding(base['embedding']))
if proy.get('disponible'):
    perfil['silueta_umap'] = proy['separacion_silueta']
    xs, ys = zip(*proy['coordenadas'])
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(xs, ys, c=range(len(xs)), cmap='viridis', s=14, alpha=0.85)
    plt.colorbar(sc, label='orden del fragmento en el documento')
    ax.set_title(f"Mapa semantico (silueta={proy['separacion_silueta']}, via {proy.get('metodo')})")
    ax.set_xticks([]); ax.set_yticks([]); plt.show()


## 5. La huella lexica
Jaccard de vocabulario entre tramos del documento. Bloques calientes = zonas que reusan las mismas
palabras (alli bm25 discrimina poco DENTRO del bloque, pero mucho ENTRE bloques).


In [ ]:
def _vocab(t): return set(re.findall(r'[a-záéíóúüña-z]+', t.lower()))
paginas = [d.page_content or '' for d in docs]
n_seg = min(24, max(4, len(paginas)))
paso = len(paginas) / n_seg
segs = [' '.join(paginas[int(i*paso):int((i+1)*paso)]) for i in range(n_seg)]
V = [_vocab(s) for s in segs]
M = np.zeros((n_seg, n_seg))
for i in range(n_seg):
    for j in range(n_seg):
        u = V[i] | V[j]
        M[i, j] = len(V[i] & V[j]) / len(u) if u else 0
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(M, cmap='magma')
plt.colorbar(im, label='Jaccard de vocabulario')
ax.set_title('La huella lexica del documento (tramo vs tramo)')
ax.set_xlabel('tramo'); ax.set_ylabel('tramo'); plt.show()
print('repetitividad global (perfil):', perfil.get('repetitividad_lexica'))


## 6. ¿Y que diria la memoria de este documento?
Los priors por perfil ANTES de gastar un solo experimento: la apuesta informada del sistema.


In [ ]:
from rag_lab.supervisor.meta import CLAVE_DECISION, NOMBRES_BANCOS
for banco in NOMBRES_BANCOS:
    pr = mem.priors(perfil, banco, CLAVE_DECISION[banco])
    if pr.get('disponible'):
        print(f"  {banco:10}: tipico={pr['ganador']} (acuerdo {pr['acuerdo']:.0%} de "
              f"{pr['n_vecinos']} vecinos, sim {pr['sim_media']}) valor~{pr['valor_esperado']}")
    else:
        print(f'  {banco:10}: sin vecinos comparables')
